In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
import requests
import zipfile
from pathlib import Path

url = "https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip"

data_path = Path("data")

zip_path = data_path / "pizza_steak_sushi.zip"

data_path.mkdir(exist_ok=True)

response = requests.get(url)

with open(zip_path, "wb") as f:
    f.write(response.content)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(data_path)

print("Dataset downloaded and extracted.")

Dataset downloaded and extracted.


In [3]:
#transform = transforms.Compose([
   # transforms.Resize((64, 64)),
    #transforms.ToTensor()
#])

train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [4]:
train_dataset = datasets.ImageFolder(
    "data/train",
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    "data/test",
    transform=test_transform
)

In [5]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [6]:
print(train_dataset.classes)

['pizza', 'steak', 'sushi']


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

cuda


In [10]:
"""
class SimpleCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 16 * 16, 128),
            nn.ReLU(),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
        """

'\nclass SimpleCNN(nn.Module):\n\n    def __init__(self):\n        super().__init__()\n\n        self.features = nn.Sequential(\n            nn.Conv2d(3, 16, kernel_size=3, padding=1),\n            nn.ReLU(),\n            nn.MaxPool2d(2),\n\n            nn.Conv2d(16, 32, kernel_size=3, padding=1),\n            nn.ReLU(),\n            nn.MaxPool2d(2)\n        )\n\n        self.classifier = nn.Sequential(\n            nn.Flatten(),\n            nn.Linear(32 * 16 * 16, 128),\n            nn.ReLU(),\n            nn.Linear(128, 3)\n        )\n\n    def forward(self, x):\n        x = self.features(x)\n        x = self.classifier(x)\n        return x\n        '

In [11]:
class BetterCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 3)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [12]:

"""
model = SimpleCNN().to(device)

print(model)
"""

'\nmodel = SimpleCNN().to(device)\n\nprint(model)\n'

In [13]:
model = BetterCNN().to(device)

In [18]:
loss_fn = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [19]:
epochs = 10

for epoch in range(epochs):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = loss_fn(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

    accuracy = 100 * correct / total

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Loss: {total_loss:.4f} | "
        f"Accuracy: {accuracy:.2f}%"
    )

Epoch 1/10 | Loss: 4.4926 | Accuracy: 71.11%
Epoch 2/10 | Loss: 6.3819 | Accuracy: 76.44%
Epoch 3/10 | Loss: 4.0512 | Accuracy: 78.67%
Epoch 4/10 | Loss: 4.2225 | Accuracy: 76.89%
Epoch 5/10 | Loss: 3.7736 | Accuracy: 76.00%
Epoch 6/10 | Loss: 8.4315 | Accuracy: 78.22%
Epoch 7/10 | Loss: 4.9219 | Accuracy: 80.89%
Epoch 8/10 | Loss: 3.4808 | Accuracy: 82.22%
Epoch 9/10 | Loss: 3.5235 | Accuracy: 80.00%
Epoch 10/10 | Loss: 4.4183 | Accuracy: 82.67%


In [20]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 62.67%


In [21]:
torch.save(
    model.state_dict(),
    "pizza_steak_sushi_model.pth"
)

print("Model saved.")

Model saved.
